In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, glob, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, recall_score
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

RESULTS_DIR = "/content/drive/MyDrive/CNN_results"
DIS_DIR = os.path.join(RESULTS_DIR, "final_densenet121")        # notebook 06a (disease detection)
SEV_DIR = os.path.join(RESULTS_DIR, "severity_densenet121")     # notebook 07 (severity)
OUT_DIR = os.path.join(RESULTS_DIR, "combined_final")
os.makedirs(OUT_DIR, exist_ok=True)
SEV_FILE = glob.glob("/content/drive/MyDrive/**/Severity_Labels_PerDisease.xlsx", recursive=True)[0]
MANIFEST = glob.glob("/content/drive/MyDrive/**/Image_Manifest_MultiLabel.csv", recursive=True)[0]

DIS_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]
FECD_GROUP = ["FECD", "PolyPleo", "LowCD"]

dis = pd.read_csv(os.path.join(DIS_DIR, "test_predictions_all.csv"))
sev_pred = pd.read_csv(os.path.join(SEV_DIR, "test_predictions_severity.csv"))
sev_oof = pd.read_csv(os.path.join(SEV_DIR, "oof_predictions.csv"))
sev_sum = json.load(open(os.path.join(SEV_DIR, "summary_severity.json")))
labels = pd.read_excel(SEV_FILE, sheet_name="Severity_Labels", keep_default_na=False, na_values=[""])
man = pd.read_csv(MANIFEST)

print("06a disease predictions :", dis.shape)
print("07 severity predictions :", sev_pred.shape)
print("07 severity thresholds  :", {k: round(v, 3) for k, v in sev_sum["severity_thresholds"].items()})

06a disease predictions : (208, 38)
07 severity predictions : (208, 18)
07 severity thresholds  : {'FECDgroup': 0.434, 'Folds': 0.277}


In [4]:
t = (dis[["Tissue_ID", "Combination"] + DIS_COLS
         + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]]
     .merge(sev_pred[["Tissue_ID", "Label_Origin", "PSev_FECDgroup", "PSev_Folds"]], on="Tissue_ID")
     .merge(labels[["Tissue_ID", "FECDgroup_Sev", "Folds_Sev", "Guttata_Sev", "Overall_Sev"]], on="Tissue_ID")
     .merge(man[["Tissue_ID", "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"]], on="Tissue_ID", how="left"))
print("Test corneas combined:", len(t), "(should be 208)")
t.head(3)

Test corneas combined: 208 (should be 208)


,Tissue_ID,Combination,FECD,Guttata,Folds,PolyPleo,LowCD,P_Fus_FECD,P_Fus_Guttata,P_Fus_Folds,P_Fus_PolyPleo,P_Fus_LowCD,Pred_Fus_FECD,Pred_Fus_Guttata,Pred_Fus_Folds,Pred_Fus_PolyPleo,Pred_Fus_LowCD,Label_Origin,PSev_FECDgroup,PSev_Folds,FECDgroup_Sev,Folds_Sev,Guttata_Sev,Overall_Sev,ECD_CD_cells_per_mm2,CV,HEX_percent,Donor_Age
0,NEB-2020-09-12663 R,Normal,0,0,0,0,0,0.001649,0.480957,0.833008,0.002243,0.001682,0,1,1,0,0,Original,0.157349,0.901855,None,None,None,None,2740.0,41.0,56.0,72.0
1,NEB-2020-09-12667 R,Normal,0,0,0,0,0,0.000995,0.034851,0.824219,0.002396,0.001929,0,0,1,0,0,Original,0.006058,0.011536,None,None,None,None,2833.0,34.0,61.0,69.0
2,NEB-2020-09-12672 R,Normal,0,0,0,0,0,0.491699,0.037964,0.676270,0.581055,0.627930,1,0,1,1,1,Original,0.036133,0.029907,None,None,None,None,2232.0,38.0,52.0,59.0


In [5]:
tr = sev_oof.merge(labels[["Tissue_ID", "Folds_Sev"]], on="Tissue_ID")
tr = tr[tr["Folds_Sev"].isin(["Mild", "Moderate", "Severe"])]
y = tr["Folds_Sev"].isin(["Moderate", "Severe"]).astype(int).values
p = tr["PSev_Folds"].values
rows = []
for thr in [sev_sum["severity_thresholds"]["Folds"], 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    yhat = (p >= thr).astype(int)
    rows.append({"Cut-off": round(thr, 3), "Sensitivity (catch Mod/Sev)": recall_score(y, yhat),
                 "Specificity (keep Mild as Mild)": recall_score(y, yhat, pos_label=0),
                 "Mild wrongly flagged (of 268)": int(((yhat == 1) & (y == 0)).sum()),
                 "Mod/Sev missed (of 35)": int(((yhat == 0) & (y == 1)).sum())})
print("Folds severity cut-off options (training CV, 268 Mild / 35 Mod+Sev):")
display(pd.DataFrame(rows).round(3))

Folds severity cut-off options (training CV, 268 Mild / 35 Mod+Sev):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 268),Mod/Sev missed (of 35)
0,0.277,0.914,0.687,84,3
1,0.350,0.857,0.720,75,5
2,0.400,0.771,0.761,64,8
3,0.450,0.743,0.784,58,9
4,0.500,0.686,0.821,48,11
5,0.550,0.657,0.843,42,12
6,0.600,0.629,0.851,40,13


In [6]:
FOLDS_SEV_THR = sev_sum["severity_thresholds"]["Folds"]   # change after looking at the table above (e.g. 0.40)
FECD_SEV_THR = sev_sum["severity_thresholds"]["FECDgroup"]
FECD_SEV_METHOD = "model"     # "model" = notebook 07 output | "cd_rule" = cell density < 1600 -> Mod/Sev
print(f"Folds cut-off {FOLDS_SEV_THR:.3f} | FECD cut-off {FECD_SEV_THR:.3f} | FECD method: {FECD_SEV_METHOD}")

Folds cut-off 0.277 | FECD cut-off 0.434 | FECD method: model


In [7]:
ORD = {"None": 0, "Mild": 1, "Mod/Sev": 2}

def fecd_severity(r):
    if FECD_SEV_METHOD == "cd_rule":
        return "Mod/Sev" if r["ECD_CD_cells_per_mm2"] < 1600 else "Mild"
    return "Mod/Sev" if r["PSev_FECDgroup"] >= FECD_SEV_THR else "Mild"

def cornea_report(r):
    found = [c for c in DIS_COLS if r[f"Pred_Fus_{c}"] == 1]          # diseases from notebook 06a
    sevs = []
    if any(c in found for c in FECD_GROUP): sevs.append(("FECD group", fecd_severity(r)))
    if "Folds" in found: sevs.append(("Folds", "Mod/Sev" if r["PSev_Folds"] >= FOLDS_SEV_THR else "Mild"))
    if "Guttata" in found: sevs.append(("Guttata", "Mild"))           # not gradable: all recorded cases Mild
    overall = max([s for _, s in sevs], key=lambda s: ORD[s]) if sevs else "None"
    return pd.Series({"Predicted diseases": " + ".join(found) if found else "Normal",
                      "Predicted severity": ", ".join(f"{d}: {s}" for d, s in sevs) if sevs else "-",
                      "Predicted overall": overall})

to3 = lambda s: s.astype(str).replace({"Moderate": "Mod/Sev", "Severe": "Mod/Sev", "nan": "None"})
rep = pd.concat([t, t.apply(cornea_report, axis=1)], axis=1)
rep["True overall"] = to3(rep["Overall_Sev"])

L = ["None", "Mild", "Mod/Sev"]
cm = pd.DataFrame(confusion_matrix(rep["True overall"], rep["Predicted overall"], labels=L),
                  index=[f"True {l}" for l in L], columns=[f"Pred {l}" for l in L])
print("FINAL end-to-end overall severity (test): 06a diseases + 07 severity")
display(cm)
agree = (rep["True overall"] == rep["Predicted overall"]).mean()
print(f"Exact agreement: {agree:.1%}   (notebook 07 alone was 63.0%)")
print(f"Mod/Sev corneas called Normal: {int(cm.loc['True Mod/Sev', 'Pred None'])} of {int(cm.loc['True Mod/Sev'].sum())}")

FINAL end-to-end overall severity (test): 06a diseases + 07 severity


,Pred None,Pred Mild,Pred Mod/Sev
True None,62,32,5
True Mild,9,55,30
True Mod/Sev,0,3,12


Exact agreement: 62.0%   (notebook 07 alone was 63.0%)
Mod/Sev corneas called Normal: 0 of 15


In [8]:
rows = []
for name, true_col, dis_list, prob in [("FECD group", "FECDgroup_Sev", FECD_GROUP, None), ("Folds", "Folds_Sev", ["Folds"], "PSev_Folds")]:
    m = rep[true_col].isin(["Mild", "Moderate", "Severe"]) & rep[[f"Pred_Fus_{c}" for c in dis_list]].max(axis=1).eq(1)
    yt = rep.loc[m, true_col].isin(["Moderate", "Severe"]).astype(int)
    if name == "FECD group":
        yp = rep.loc[m].apply(fecd_severity, axis=1).eq("Mod/Sev").astype(int)
    else:
        yp = (rep.loc[m, prob] >= FOLDS_SEV_THR).astype(int)
    n_true = int(rep[true_col].isin(["Mild", "Moderate", "Severe"]).sum())
    rows.append({"Disease": name, "Truly present": n_true, "Detected": int(m.sum()),
                 "Mild / Mod+Sev (detected)": f"{int((yt == 0).sum())} / {int(yt.sum())}",
                 "Severity correct": f"{(yt == yp).mean():.1%}",
                 "Sensitivity": recall_score(yt, yp, zero_division=0),
                 "Specificity": recall_score(yt, yp, pos_label=0, zero_division=0)})
print("Severity per disease on DETECTED cases (test)")
display(pd.DataFrame(rows).set_index("Disease").round(3))

Severity per disease on DETECTED cases (test)


,Truly present,Detected,Mild / Mod+Sev (detected),Severity correct,Sensitivity,Specificity
Disease,,,,,,
FECD group,27,27,20 / 7,100.0%,1.000,1.000
Folds,73,62,56 / 6,56.5%,0.833,0.536


In [9]:
print("Example final cornea reports (test):")
show = ["Tissue_ID", "Combination", "Predicted diseases", "Predicted severity", "Predicted overall", "True overall"]
display(rep[show].sample(15, random_state=3).reset_index(drop=True))
print("\nMistakes where a Mod/Sev cornea was under-called:")
display(rep.loc[(rep["True overall"] == "Mod/Sev") & (rep["Predicted overall"] != "Mod/Sev"), show])

Example final cornea reports (test):


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
0,NEB-2024-12-16642 R,Normal,Folds,Folds: Mild,Mild,None
1,NEB-2024-12-16640 R,Folds,Guttata + Folds,"Folds: Mod/Sev, Guttata: Mild",Mod/Sev,Mild
2,NEB-2023-09-15405 R,Normal,Folds,Folds: Mild,Mild,None
3,NEB-2025-10-17549 R,Guttata,Normal,-,None,Mild
4,NEB-2024-08-16285 L,Normal,Normal,-,None,None
5,NEB-2023-03-14888 R,Normal,Guttata + Folds,"Folds: Mild, Guttata: Mild",Mild,None
6,NEB-2024-08-16274 R,Folds,Folds,Folds: Mod/Sev,Mod/Sev,Mild
7,NEB-2024-08-16269 L,Folds,FECD + Guttata + PolyPleo + LowCD,"FECD group: Mild, Guttata: Mild",Mild,Mild
8,NEB-2020-11-12820 R,Normal,Normal,-,None,None
9,NEB-2023-03-14891 R,Normal,Folds,Folds: Mild,Mild,None



Mistakes where a Mod/Sev cornea was under-called:


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
74,NEB-2023-04-14936 R,Folds,Folds,Folds: Mild,Mild,Mod/Sev
160,NEB-2024-08-16275 L,Guttata + Folds,Guttata,Guttata: Mild,Mild,Mod/Sev
194,NEB-2025-10-17595 R,Guttata,Guttata,Guttata: Mild,Mild,Mod/Sev


In [10]:
keep = (["Tissue_ID", "Label_Origin", "Combination"] + DIS_COLS
        + ["FECDgroup_Sev", "Folds_Sev", "Guttata_Sev", "True overall"]
        + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]
        + ["PSev_FECDgroup", "PSev_Folds", "Predicted diseases", "Predicted severity", "Predicted overall",
           "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"])
rep[keep].to_csv(os.path.join(OUT_DIR, "Final_Cornea_Reports_test.csv"), index=False)
cm.to_csv(os.path.join(OUT_DIR, "overall_severity_confusion.csv"))
json.dump({"disease_model": "06a DenseNet121 image+numbers", "severity_model": "07 DenseNet121 multi-task",
           "folds_sev_threshold": FOLDS_SEV_THR, "fecd_sev_threshold": FECD_SEV_THR, "fecd_sev_method": FECD_SEV_METHOD,
           "overall_exact_agreement": round(float(agree), 4)},
          open(os.path.join(OUT_DIR, "combined_summary.json"), "w"), indent=2)
print("Saved to", OUT_DIR); print(os.listdir(OUT_DIR))

Saved to /content/drive/MyDrive/CNN_results/combined_final
['Final_Cornea_Reports_test.csv', 'overall_severity_confusion.csv', 'combined_summary.json']


In [11]:
from sklearn.metrics import roc_auc_score
g = dis.merge(sev_pred[["Tissue_ID", "Label_Origin"]], on="Tissue_ID")
def auc_or_nan(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan
rows = []
for org in ["All test", "Original", "Model-assisted (2023)"]:
    s = g if org == "All test" else g[g["Label_Origin"] == org]
    for c in DIS_COLS:
        rows.append({"Group": f"{org} (n={len(s)})", "Disease": c, "Cases": int(s[c].sum()),
                     "Image + numbers AUC": auc_or_nan(s[c].values, s[f"P_Fus_{c}"].values),
                     "Numbers only AUC": auc_or_nan(s[c].values, s[f"P_Num_{c}"].values)})
res = pd.DataFrame(rows)
print("Disease AUC on the SAME test predictions, split by who made the label")
display(res.pivot(index="Disease", columns="Group", values="Image + numbers AUC").round(3))
print("\nNumbers-only model, same split (for comparison)")
display(res.pivot(index="Disease", columns="Group", values="Numbers only AUC").round(3))
print("\nNumber of diseased corneas per group")
display(res.pivot(index="Disease", columns="Group", values="Cases"))
res.to_csv(os.path.join(OUT_DIR, "disease_auc_by_label_origin.csv"), index=False)

Disease AUC on the SAME test predictions, split by who made the label


Group,All test (n=208),Model-assisted (2023) (n=94),Original (n=114)
Disease,,,
FECD,0.990,1.000,0.985
Folds,0.866,0.918,0.824
Guttata,0.872,0.908,0.852
LowCD,0.978,1.000,0.955
PolyPleo,0.990,1.000,0.983



Numbers-only model, same split (for comparison)


Group,All test (n=208),Model-assisted (2023) (n=94),Original (n=114)
Disease,,,
FECD,0.991,1.000,0.982
Folds,0.648,0.685,0.618
Guttata,0.617,0.639,0.619
LowCD,0.958,1.000,0.868
PolyPleo,0.991,1.000,0.982



Number of diseased corneas per group


Group,All test (n=208),Model-assisted (2023) (n=94),Original (n=114)
Disease,,,
FECD,25,17,8
Folds,74,35,39
Guttata,19,7,12
LowCD,23,17,6
PolyPleo,26,17,9
